# Implements a random G/G/c queue generator
## Rafael Andrade & Eriky Silva & Frederico Cruz
---

Required packages

In [3]:
import random as rnd
import heapq as hq
import pandas as pd
import numpy as np
import math

import pickle
from datetime import date

## Queue generator class
---

In [4]:
class Queue:
    def __init__(self, farr, fdep, tmax, nserv=1, farr_batch=lambda: 1, fdep_batch=lambda: 1):
        self.elements = []
        self.nserv = nserv
        self.farr = farr
        self.fdep = fdep
        self.farr_batch = farr_batch
        self.fdep_batch = fdep_batch
        self.tmax = tmax

        self.queue = 0
        self.busyserv = 0
        self.time = 0
        self.events = []

        self.log_time = []
        self.log_event = []
        self.log_queue = []
        self.log_busy = []
        self.total_arrivals = 0
        self.total_departures = 0
        
        self.schedule_arr()


    def schedule_arr(self):
        arr_time = self.time + self.farr()
        hq.heappush(self.events, (arr_time, 'arr'))


    def schedule_dep(self):
        dep_time = self.time + self.fdep()
        hq.heappush(self.events, (dep_time, 'dep'))


    def try_start_service(self):
        while self.busyserv < self.nserv and self.queue > 0:
            batch = min(self.fdep_batch(), self.queue)
            self.queue -= batch
            self.total_departures += batch
            self.busyserv += 1
            self.schedule_dep()

            
    def process_arr(self):
        self.schedule_arr()
        batch = self.farr_batch()
        self.queue += batch
        self.total_arrivals += batch
        self.try_start_service()


    def process_dep(self):
        self.busyserv -= 1
        self.try_start_service()


    def log_state(self, event):
        self.log_time.append(self.time)
        self.log_event.append(event)
        self.log_queue.append(self.queue)
        self.log_busy.append(self.busyserv)


    def run(self):
        while self.events and self.time < self.tmax:
            self.time, ev = hq.heappop(self.events)

            if ev == 'arr':
                self.process_arr()
            else:
                self.process_dep()

            self.log_state(ev)

    
    def get_metrics(self):
        if len(self.log_time) < 2:
            print("Not enough data")
            return

        total_time = self.log_time[-1]

        area_q = 0.0
        area_b = 0.0
        area_s = 0.0

        for i in range(1, len(self.log_time)):
            dt = self.log_time[i] - self.log_time[i-1]

            q = self.log_queue[i-1]
            b = self.log_busy[i-1]
            s = q + b

            area_q += q * dt
            area_b += b * dt
            area_s += s * dt

        avg_q = area_q / total_time
        avg_b = area_b / total_time
        avg_s = area_s / total_time

        n_arr = self.total_arrivals
        n_dep = self.total_departures

        utilization = avg_b / self.nserv
        lmbda_eff = n_arr / total_time
        throughput = n_dep / total_time

        w = avg_s / lmbda_eff if lmbda_eff > 0 else 0
        wq = avg_q / lmbda_eff if lmbda_eff > 0 else 0

        return {
        "rho": utilization,
        "L": avg_s,
        "Lq": avg_q,
        "W": w,
        "Wq": wq,
        "avg_in_service": avg_b,
        "throughput": throughput,
        "total_time": total_time,
        "arrivals": n_arr,
        "departures": n_dep
    }

    def print_metrics(self):
        metrics = self.get_metrics()
        print("=== Queue Metrics ===")
        print(f"Rho: {metrics['rho']:.4f}")
        print(f"L: {metrics['L']:.4f}")
        print(f"Lq: {metrics['Lq']:.4f}")
        print(f"W: {metrics['W']:.4f}")
        print(f"Wq: {metrics['Wq']:.4f}")
        print(f"Avg in service: {metrics['avg_in_service']:.4f}")
        print(f"Throughput: {metrics['throughput']:.4f}")
        print(f"Total time: {metrics['total_time']:.4f}")
        print(f"Arrivals: {metrics['arrivals']}")
        print(f"Departures: {metrics['departures']}")

    
    def print_log(self):
        return pd.DataFrame({
            'time': self.log_time,
            'event': self.log_event,
            'queue': self.log_queue,
            'busy': self.log_busy
        })


## Simulating queues  (todo)
---

#### Monte Carlo functions

In [5]:
def mc_queue(farr, fdep, tmax, nserv, farr_batch, fdep_batch, rep_mc=100):
    np.random.seed(2025)
    res = []
    for _ in range(rep_mc):
        q = Queue(farr, fdep, tmax, nserv, farr_batch, fdep_batch)
        q.run()
        m = q.get_metrics()
        res.append([m['Wq'], m['Lq'], m['rho']])
    
    df = pd.DataFrame(res, columns=['Wq', 'Lq', 'Rho'])
    return df.mean().add_suffix('_mean').to_dict() | df.var().add_suffix('_var').to_dict()


def tab_mc_queue(lambdas, mus, tmaxs, farr, fdep, nserv, farr_batch, fdep_batch, rep_mc=100):
    tab = []
    total = len(tmaxs) * len(lambdas)
    k = 0
    for lmb, mu in zip(lambdas, mus):
        for tmax in tmaxs:
            k += 1
            stats = mc_queue(farr(lmb), fdep(mu), tmax, nserv, farr_batch, fdep_batch, rep_mc)
            tab.append({'lambda': lmb, 'mu': mu, 'tmax': tmax} | stats)
            print(f"\r{100*k/total:.1f}%", end="")
    return pd.DataFrame(tab)

### Parameters

In [19]:
arr_batchs = np.array([1, 2, 3])
dep_batch = np.array([1, 2, 3])
tmaxs = [100, 500, 1000]
lambdas = np.repeat(1.0, 5) # lambda can always be taken as 1
mus = np.linspace(0.5, 2.0, 5)
rep_mc = 100

### Simulating M-M-1

#### Simulation

In [10]:
sim_MM1 = tab_mc_queue(
    lambdas = lambdas,
    mus = mus,
    tmaxs = tmaxs,
    farr = lambda lmbda: lambda: np.random.exponential(1/lmbda), # so we have a function for each lambda
    fdep = lambda mu: lambda: np.random.exponential(1/mu),
    nserv = 1,
    farr_batch = lambda: 1,
    fdep_batch = lambda: 1,
    rep_mc = rep_mc,
)

100.0%

In [13]:
sim_MM1.head(10)

,lambda,mu,tmax,Wq_mean,Lq_mean,Rho_mean,Wq_var,Lq_var,Rho_var
0,1.0,0.500,100,24.565379,24.726006,0.978742,30.942358,49.637498,0.000405
1,1.0,0.500,500,125.139508,124.955685,0.996530,149.540941,256.740839,0.000011
2,1.0,0.500,1000,248.572432,247.406346,0.998067,346.908285,522.213499,0.000004
3,1.0,0.875,100,10.180725,10.551397,0.934027,26.625259,33.522689,0.003778
4,1.0,0.875,500,38.454805,38.649821,0.985913,161.710127,189.810102,0.000215
5,1.0,0.875,1000,67.517256,67.717230,0.991852,480.025318,545.973203,0.000074
6,1.0,1.250,100,2.478862,2.592805,0.762556,3.000606,3.894361,0.012551
7,1.0,1.250,500,3.132140,3.149098,0.790573,2.393760,2.819652,0.002508
8,1.0,1.250,1000,3.143398,3.146899,0.797026,1.109716,1.232536,0.001080
9,1.0,1.625,100,0.914854,0.942980,0.603568,0.302692,0.412752,0.006876


### Simulating Mx-M-1 (todo)

In [23]:
# sim_MxM1 = tab_mc_queue(
#     lambdas = lambdas,
#     mus = mus,
#     tmaxs = tmaxs,
#     farr = lambda lmbda: lambda: np.random.exponential(1/lmbda), # so we have a function for each lambda
#     fdep = lambda mu: lambda: np.random.exponential(1/mu),
#     nserv = 1,
#     farr_batch = lambda: arr_batchs[0],
#     fdep_batch = lambda: 1,
#     rep_mc = rep_mc,
# )

In [ ]:
sim_MxM1 = pd.DataFrame()

for batch in arr_batchs:
    sim = tab_mc_queue(
        lambdas = lambdas,
        mus = mus,
        tmaxs = tmaxs,
        farr = lambda lmbda: lambda: np.random.exponential(1/lmbda),
        fdep = lambda mu: lambda: np.random.exponential(1/mu),
        nserv = 1,
        farr_batch = lambda: batch,
        fdep_batch = lambda: 1,
        rep_mc = rep_mc,
    )

    sim["batch"] = batch
    sim_MxM1 = pd.concat([sim_MxM1, sim], ignore_index=True)

100.0%

In [ ]:
sim_MxM1.head(10)

,lambda,mu,tmax,Wq_mean,Lq_mean,Rho_mean,Wq_var,Lq_var,Rho_var,batch
0,1.0,0.500,100,24.565379,24.726006,0.978742,30.942358,49.637498,0.000405,1
1,1.0,0.500,500,125.139508,124.955685,0.996530,149.540941,256.740839,0.000011,1
2,1.0,0.500,1000,248.572432,247.406346,0.998067,346.908285,522.213499,0.000004,1
3,1.0,0.875,100,10.180725,10.551397,0.934027,26.625259,33.522689,0.003778,1
4,1.0,0.875,500,38.454805,38.649821,0.985913,161.710127,189.810102,0.000215,1
5,1.0,0.875,1000,67.517256,67.717230,0.991852,480.025318,545.973203,0.000074,1
6,1.0,1.250,100,2.478862,2.592805,0.762556,3.000606,3.894361,0.012551,1
7,1.0,1.250,500,3.132140,3.149098,0.790573,2.393760,2.819652,0.002508,1
8,1.0,1.250,1000,3.143398,3.146899,0.797026,1.109716,1.232536,0.001080,1
9,1.0,1.625,100,0.914854,0.942980,0.603568,0.302692,0.412752,0.006876,1


In [ ]:
#print(mxm1_metrics(lambdas_global[0] / arr_batch, arr_batch, mus_global[0]))

### Simulating M-Mx-1  (todo)

In [ ]:
sim_MMx1 = pd.DataFrame()

for batch in dep_batchs:
sim = tab_mc_queue(
    lambdas = lambdas,
    mus = mus,
    tmaxs = tmaxs,
    farr = lambda lmbda: lambda: np.random.exponential(1/lmbda), # so we have a function for each lambda
    fdep = lambda mu: lambda: np.random.exponential(1/mu),
    nserv = 1,
    farr_batch = lambda: 1,
    fdep_batch = lambda: dep_batch,
    rep_mc = rep_mc,
)

100.0%

In [ ]:
sim_MMx1.head(10)

,lambda,mu,tmax,Wq_mean,Lq_mean,Rho_mean,Wq_var,Lq_var,Rho_var
0,0.5,0.666667,100,0.894269,0.457932,0.562791,0.174104,0.057704,0.007536
1,0.5,0.666667,500,0.924834,0.457550,0.565058,0.035296,0.011506,0.001452
2,0.5,0.666667,1000,0.961943,0.477607,0.572056,0.025738,0.007451,0.000764
3,1.0,0.666667,100,1.586130,1.571060,0.806681,0.436512,0.479031,0.003603
4,1.0,0.666667,500,1.843452,1.829391,0.825431,0.137598,0.159297,0.000848
5,1.0,0.666667,1000,1.833430,1.820393,0.826235,0.070440,0.075449,0.000377
6,1.5,0.666667,100,2.843039,4.312852,0.924123,2.335728,6.675727,0.001586
7,1.5,0.666667,500,3.839372,5.732797,0.936640,1.606442,3.916438,0.000365
8,1.5,0.666667,1000,3.848485,5.752827,0.939384,0.680242,1.663640,0.000165


### Save results and clear variables

In [ ]:
filename = f"sim_{date.today()}.pkl"

results = {
    'lambdas': lambdas,
    'mus': mus,
    'arr_batch': arr_batchs,
    'dep_batch': dep_batch,
    'tmaxs': tmaxs,
    'rep_mc': rep_mc,
    'sim_MM1': sim_MM1,
    'sim_MxM1': sim_MxM1,
    'sim_MMx1': sim_MMx1,
}

with open(filename, 'wb') as f:
    pickle.dump(results, f)

del lambdas, mus, tmaxs, rep_mc, arr_batchs, dep_batch, filename, results

## Testing queue generator
---

### M-M-c test

Theoretical metrics

In [ ]:
def mmc_metrics(lmbda, mu, c):
    rho = lmbda / (c * mu)
    s   = sum((lmbda / mu)**n / math.factorial(n) for n in range(c))
    t   = (lmbda / mu)**c / (math.factorial(c) * (1 - rho))
    P0  = 1 / (s + t)
    Pw  = t * P0
    Wq  = Pw / (c * mu - lmbda)
    W   = Wq + 1 / mu
    Lq  = lmbda * Wq
    L   = lmbda * W
    
    return {
        'rho': rho,
        'Pw': Pw,
        'Wq': Wq,
        'W': W,
        'Lq': Lq,
        'L': L
    }

In [ ]:
nserv  = 3
rho    = 0.5
lmbda  = 2.0
mu     = lmbda / (nserv * rho)

rho, Pw, Wq, W, Lq, L = mmc_metrics(lmbda, mu, nserv)

print("Rho:", round(rho, 4))
print("W  :", round(W, 4))
print("Wq :", round(Wq, 4))
print("Lq :", round(Lq, 4))
print("L  :", round(L, 4))
print("Pw :", round(Pw, 4))

Simulated metrics

In [ ]:
farr = lambda: rnd.expovariate(lmbda)
fdep = lambda: rnd.expovariate(mu)

q = Queue(nserv=nserv, farr=farr, fdep=fdep, tmax=10000)

q.run()
q.print_metrics()

### Mx-M-1 test

Theoretical metrics

In [ ]:
def mxm1_metrics(lmbda, batch_size, mu_global):
    lmbda_global = lmbda * batch_size
    rho       = lmbda_global / mu_global
    EX2       = batch_size**2
    L         = (rho + (lmbda / mu_global) * EX2) / (2 * (1 - rho))
    W         = L / lmbda_global
    Wq        = W - 1 / mu_global
    Lq        = lmbda_global * Wq
    return {
        'lambda_global': float(lmbda_global),
        'rho': float(rho),
        'L': float(L),
        'Lq': float(Lq),
        'W': float(W),
        'Wq': float(Wq)
    }

In [ ]:
lmbda_val = 0.2
batch     = 5
mu_val    = 2.0

lmbda_eff, rho, L, Lq, W, Wq = mxm1_metrics(lmbda_val, batch, mu_val)

print("rho       :", round(rho, 4))
print("L         :", round(L, 4))
print("Lq        :", round(Lq, 4))
print("W         :", round(W, 4))
print("Wq        :", round(Wq, 4))
print("lambda_eff:", round(lmbda_eff, 4))

Simulated metrics

In [ ]:
farr = lambda: rnd.expovariate(lmbda_val)
fdep = lambda: rnd.expovariate(mu_val)
farr_batch=lambda: batch
fdep_batch=lambda: 1

q = Queue(nserv=1, farr=farr, fdep=fdep, farr_batch=farr_batch, fdep_batch=fdep_batch, tmax=100000)

q.run()
q.print_metrics()

### M-G-1 test

Theoretical metrics

In [ ]:
def mg1_metrics(lmbda, ES, VarS):
    rho = lmbda * ES
    Cv2 = VarS / (ES**2)

    Lq  = ((1 + Cv2) / 2) * (rho**2 / (1 - rho))
    W   = ((1 + Cv2) / 2) * (rho / (1/ES - lmbda)) + ES
    L   = Lq + rho
    Wq  = W - ES

    return rho, L, Lq, W, Wq

In [ ]:
lmbda = 0.8
a, b  = 0.7, 1.7

ES    = (a + b) / 2
VarS  = (b - a)**2 / 12

rho, L, Lq, W, Wq = mg1_metrics(lmbda, ES, VarS)

print("rho:", round(rho, 4))
print("L  :", round(L, 4))
print("Lq :", round(Lq, 4))
print("W  :", round(W, 4))
print("Wq :", round(Wq, 4))

rho: 0.96
L  : 13.1467
Lq : 12.1867
W  : 16.4333
Wq : 15.2333


Simulated metrics

In [ ]:
q = Queue(
    farr = lambda: rnd.expovariate(lmbda),
    fdep = lambda: rnd.uniform(a, b),
    tmax = 200000
)

q.run()
q.print_metrics()

=== Queue Metrics ===
Rho: 0.9611
L: 13.7609
Lq: 12.7998
W: 17.1787
Wq: 15.9789
Avg in service: 0.9611
Throughput: 0.8010
Total time: 200000.2450
Arrivals: 160209
Departures: 160203
